# Foundry IQ knowledge base with Web IQ MCP server

In Part 1, the knowledge base only queried pre-built search indexes. Now you'll add a third source type: an **MCP Server Knowledge Source** that connects to the new Web IQ service.

- **MCP Server Knowledge Source**: connects to any remote MCP endpoint. The KB calls it at query time like a tool.
- **Web IQ**: Microsoft's web grounding service exposed as an MCP server. Returns ranked, cited web results.

## Step 1: Set up environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has *almost* everything you need: Azure AI Search endpoints, Azure OpenAI credentials, and model configuration.

‼️ You also need a `WEB_IQ_KEY` variable. This is only available for allow-listed customers at this time.

```shell
WEB_IQ_KEY="some-value-here"
```

Then run the cell below to load in those variables:

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
WEB_IQ_KEY = os.environ["WEB_IQ_KEY"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


## Step 2: Create three knowledge sources

For this knowledge base, you'll first create three knowledge sources - the same two index-based sources as you made in the first notebook, plus an additional knowledge source for the Web IQ MCP server:

* `healthdocs-knowledge-source`: Points to the `healthdocs` search index
* `hrdocs-knowledge-source`: Points to the `hrdocs` search index
* `web-knowledge-source`: Points to the remote MCP server for the Web IQ service, using key-based authentication and `web` tool.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    McpServerAutoOutputParsing,
    McpServerKnowledgeSource,
    McpServerKnowledgeSourceParameters,
    McpServerStoredHeadersAuthentication,
    McpServerStoredHeadersParameters,
    McpServerTool,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WEB_KNOWLEDGE_SOURCE_NAME = "web-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME, WEB_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Contoso HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Contoso health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

web_knowledge_source = McpServerKnowledgeSource(
    name=WEB_KNOWLEDGE_SOURCE_NAME,
    description="Web IQ (live web search)",
    mcp_server_parameters=McpServerKnowledgeSourceParameters(
        server_url="https://api.microsoft.ai/v3/mcp",
        authentication=McpServerStoredHeadersAuthentication(
            stored_headers_parameters=McpServerStoredHeadersParameters(
                {"headers": {"x-apikey": WEB_IQ_KEY}}
            )
        ),
        tools=[McpServerTool(name="web", output_parsing=McpServerAutoOutputParsing())],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=web_knowledge_source)
print("Knowledge sources created")


## Step 3: Create the multi-source + web knowledge base

A knowledge base is the orchestration layer that combines:

1. Your data sources (the knowledge sources from Step 2)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format response

For this notebook, we are using an `outputMode` of "answerSynthesis" so that the attached LLM can also answer the original user query. We are also using a `retrievalReasoningEffort` of "low", which means that the LLM will be used for query planning and knowledge source selection.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-web-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Contoso knowledge base combining HR and health documents with access to web search.",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use the HR and health indexes for company policy questions. Use Web IQ web tool for context from public webpages.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## Step 4: Query the knowledge base

Ask a question where one half needs internal data and the other needs current external benchmarks:

* _"What mental health coverage does Contoso offer?"_ → should come from `healthdocs`
* _"What are industry benchmarks for mental health benefits at tech companies?"_ → should come from the **web** via Web IQ

The knowledge base uses agentic retrieval to:

1. Analyze the query and recognize two distinct topic domains
2. Decompose it into focused sub-queries
3. Route the internal question to the search indexes and the benchmark question to the web
4. Run searches concurrently across all three sources
5. Merge results using semantic re-ranking

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    KnowledgeSourceParams,
    SearchIndexKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)
web_knowledge_source_params = KnowledgeSourceParams(
    knowledge_source_name=WEB_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
    kind="mcpServer",
)

question = (
    "A new hire wants to know if Contoso's benefits are competitive. "
    "What mental health coverage does Contoso offer according to our internal health plan docs? "
    "And search the web for current industry benchmarks for mental health benefits at tech companies."
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params, web_knowledge_source_params],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
display(Markdown(result.response[0].content[0].text))


### Review the activity log

For this activity log, you will see an "mcpServer" step for the call made to Web IQ, along with "searchIndex" steps for each query sent to a search index.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

For the Web IQ knowledge source, the references have `type: "mcpServer"`. The article title, content, and URL are inside `sourceData.content`, which is a JSON string.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))